# 03 — Data Integration

Load cleaned datasets and:
- Discover value-based join connections across all 14 tables
- Confirm pairwise linkages (ACH- IDs, PR- IDs, ENSG IDs, CVCL accessions)
- Save integrated / processed outputs to `data/processed/`

In [ ]:
import sys, os
#sys.path.insert(0, os.path.join(os.path.dirname('__file__'), '..', 'scripts'))
sys.path.insert(0, "/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/src/scripts")

import re
import pandas as pd
from itertools import combinations
from src.scripts.data_utils import load_clean_parquets
from src.scripts.export_data import export_processed_csvs, export_clean_report

## Load Cleaned Data

In [ ]:
tables = load_clean_parquets()

## Value-Based Join Discovery

Find connections between tables by comparing actual values in candidate key columns.

In [ ]:
ID_HINTS = re.compile(
    r"(id|name|accession|gsm|cvcl|ach|pr-|profile|model|gene|cell|symbol|ensembl|ccle|rrid|cosmic)",
    re.IGNORECASE,
)

def get_key_columns(df: pd.DataFrame, sample_n: int = 200_000) -> list:
    if len(df) > sample_n:
        df = df.sample(sample_n, random_state=0)
    return [
        col for col in df.columns
        if (bool(ID_HINTS.search(str(col))) or df[col].dtype == object)
        and df[col].nunique(dropna=True) >= 2
    ]


def build_fingerprints(tables: dict, sample_n: int = 200_000) -> dict:
    fingerprints = {}
    for name, df in tables.items():
        key_cols = get_key_columns(df, sample_n=sample_n)
        df_use = df.sample(sample_n, random_state=0) if len(df) > sample_n else df
        col_sets = {
            col: set(df_use[col].dropna().astype(str).str.strip().str.lower().unique())
            for col in key_cols
        }
        fingerprints[name] = col_sets
        print(f"{name:25s}: {len(col_sets)} candidate key columns")
    return fingerprints


def find_connections(fingerprints: dict, min_overlap: int = 5, min_pct: float = 5.0) -> pd.DataFrame:
    rows = []
    for t1, t2 in combinations(fingerprints.keys(), 2):
        for col1, set1 in fingerprints[t1].items():
            for col2, set2 in fingerprints[t2].items():
                if not set1 or not set2:
                    continue
                inter = set1 & set2
                n = len(inter)
                if n < min_overlap:
                    continue
                pct = n / min(len(set1), len(set2)) * 100
                if pct < min_pct:
                    continue
                rows.append({
                    "table_1": t1, "column_1": col1,
                    "table_2": t2, "column_2": col2,
                    "n_overlap": n,
                    "n_distinct_1": len(set1), "n_distinct_2": len(set2),
                    "pct_of_smaller": round(pct, 1),
                    "example_shared": list(inter)[:3],
                })
    result = pd.DataFrame(rows)
    if not result.empty:
        result = result.sort_values(["pct_of_smaller", "n_overlap"], ascending=False).reset_index(drop=True)
    return result

In [ ]:
print("Building value fingerprints...")
fingerprints = build_fingerprints(tables)

In [ ]:
print("Finding value-based connections...")
connections = find_connections(fingerprints, min_overlap=5, min_pct=5.0)
print(f"Found {len(connections)} candidate join connections.")
connections

## Confirm Pairwise Key Linkages

In [ ]:
sample_info    = tables["sample_info"]
depmap_profiles= tables["depmap_profiles"]
mutations      = tables["mutations"]
depmap_expr    = tables["depmap_expr"]

# ACH- linkage: sample_info <-> depmap_profiles
achs_si   = set(sample_info["depmap_id"].dropna())
achs_prof = set(depmap_profiles["modelid"].dropna())
print(f"sample_info ACH IDs:       {len(achs_si):>5,}")
print(f"depmap_profiles ACH IDs:   {len(achs_prof):>5,}")
print(f"Overlap:                   {len(achs_si & achs_prof):>5,}")
print()

# PR- linkage: depmap_profiles <-> mutations
pr_prof = set(depmap_profiles["profileid"].dropna())
pr_mut  = set(mutations["profileid"].dropna()) if "profileid" in mutations.columns else set()
print(f"depmap_profiles PR IDs:    {len(pr_prof):>5,}")
print(f"mutations PR IDs:          {len(pr_mut):>5,}")
print(f"Overlap:                   {len(pr_prof & pr_mut):>5,}")

## Save Column-Level Data Quality Report

In [ ]:
export_clean_report(tables)

## Export Processed Files to data/processed/

In [ ]:
# Export all cleaned tables as CSV to the processed directory
export_processed_csvs(tables)